# Lab 1: The HuggingFace Ecosystem — A Guided Tour (SOLUTIONS)
## CCI Session 4

**Duration:** 10 minutes

### Clinical Scenario
> KHCC's AI Office needs to evaluate open-source clinical NLP models for potential deployment — models that can run locally without sending patient data to external APIs. Your task is to explore the HuggingFace ecosystem and identify promising clinical models.

### Objective
- Navigate the HuggingFace Model Hub
- Read and evaluate model cards
- Test models using the Inference API
- Understand licensing for clinical use

### Setup
Run the cell below to install the required packages.

In [ ]:
# Install required packages
!pip install -q transformers datasets huggingface_hub

---
### Cell 1: Exploring the Hub Programmatically
HuggingFace hosts thousands of models. Let's search for clinical and biomedical models using the Python API.

In [ ]:
# === CELL 1: BROWSE THE HUB ===
from huggingface_hub import HfApi
api = HfApi()

# Search for models with the tag "medical"
print("=== Top Medical Models by Downloads ===")
medical_models = api.list_models(search="medical", sort="downloads", limit=10)
for model in medical_models:
    print(f"  Model: {model.id}")
    print(f"  Downloads: {model.downloads}")
    print(f"  Tags: {model.tags[:5] if model.tags else 'N/A'}")
    print()

# Search for NER models in the biomedical domain
print("\n=== Biomedical NER Models ===")
ner_models = api.list_models(search="biomedical ner", sort="downloads", limit=10)
for model in ner_models:
    print(f"  Model: {model.id}")
    print(f"  Downloads: {model.downloads}")
    print(f"  Pipeline: {model.pipeline_tag}")
    print()

---
### Cell 2: Reading Model Cards
Model cards describe what a model was trained on, its intended use, and its limitations. This is critical for clinical deployment decisions.

In [ ]:
# === CELL 2: MODEL CARDS ===
from huggingface_hub import model_info

# Get info for the biomedical NER model
print("=== d4data/biomedical-ner-all ===")
info1 = model_info("d4data/biomedical-ner-all")
print(f"  Model ID: {info1.id}")
print(f"  Pipeline Tag: {info1.pipeline_tag}")
print(f"  Downloads: {info1.downloads}")
print(f"  License: {info1.card_data.license if info1.card_data else 'N/A'}")
print(f"  Tags: {info1.tags}")

# Compare with Bio_ClinicalBERT
print("\n=== emilyalsentzer/Bio_ClinicalBERT ===")
info2 = model_info("emilyalsentzer/Bio_ClinicalBERT")
print(f"  Model ID: {info2.id}")
print(f"  Pipeline Tag: {info2.pipeline_tag}")
print(f"  Downloads: {info2.downloads}")
print(f"  License: {info2.card_data.license if info2.card_data else 'N/A'}")
print(f"  Tags: {info2.tags}")

print("\n--- Key Differences ---")
print("d4data/biomedical-ner-all: Fine-tuned for NER on multiple biomedical datasets.")
print("Bio_ClinicalBERT: Pre-trained on MIMIC-III clinical notes; a foundation model")
print("that needs fine-tuning for specific tasks.")

---
### Cell 3: Exploring Datasets
HuggingFace also hosts datasets. Let's explore a clinical dataset to see what's available for training or evaluation.

In [ ]:
# === CELL 3: DATASETS HUB ===
from datasets import load_dataset

# Load a small preview of a clinical dataset
ds = load_dataset("tgrex6/mimic-cxr-reports-summarization", split="train[:5]")

print("=== Dataset Preview ===")
print(f"Column names: {ds.column_names}")
print(f"Number of examples loaded: {len(ds)}")

print("\n=== First Example ===")
for key, value in ds[0].items():
    print(f"  {key}: {str(value)[:200]}..." if len(str(value)) > 200 else f"  {key}: {value}")

# Check the full dataset size
print("\n=== Full Dataset Info ===")
ds_full_info = load_dataset("tgrex6/mimic-cxr-reports-summarization", split="train")
print(f"Total training examples: {len(ds_full_info)}")
print(f"Columns: {ds_full_info.column_names}")

---
### Cell 4: Quick Inference Test
Let's test a biomedical language model and compare it with a general-purpose model to see if domain-specific training makes a difference.

In [ ]:
# === CELL 4: INFERENCE API ===
from transformers import pipeline

# Load a fill-mask pipeline with a biomedical model
print("Loading biomedical model...")
unmasker = pipeline("fill-mask", model="microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract")

prompt = "The patient was diagnosed with [MASK] cancer."
print(f"\nPrompt: {prompt}")

print("\n=== Biomedical Model Predictions ===")
results_bio = unmasker(prompt)
for r in results_bio:
    print(f"  {r['token_str']:20s} (score: {r['score']:.4f})")

# Compare with a general model
print("\nLoading general model...")
unmasker_general = pipeline("fill-mask", model="bert-base-uncased")

print("\n=== General Model Predictions ===")
results_gen = unmasker_general(prompt)
for r in results_gen:
    print(f"  {r['token_str']:20s} (score: {r['score']:.4f})")

print("\n--- Observation ---")
print("The biomedical model predicts specific cancer types (e.g., breast, lung, prostate)")
print("while the general model may predict less clinically relevant terms.")

---
### Cell 5: Licensing Check
Before deploying any model in a clinical setting, we must verify its license allows commercial/clinical use.

In [ ]:
# === CELL 5: LICENSING ===
from huggingface_hub import model_info

models_to_check = [
    "d4data/biomedical-ner-all",
    "google/flan-t5-small",
    "meta-llama/Llama-3.1-8B",
]

print("=== License Check for Clinical Use ===")
print(f"{'Model':<35} {'License':<25} {'Clinical Use?'}")
print("-" * 80)

for model_name in models_to_check:
    try:
        info = model_info(model_name)
        license_str = info.card_data.license if info.card_data and info.card_data.license else "Not specified"
    except Exception as e:
        license_str = f"Error: {str(e)[:30]}"

    # Determine clinical use feasibility based on license
    license_lower = license_str.lower()
    if license_lower in ["apache-2.0", "mit", "bsd-3-clause"]:
        clinical_use = "Yes"
    elif "llama" in license_lower or "community" in license_lower:
        clinical_use = "With restrictions"
    elif license_lower in ["not specified"]:
        clinical_use = "Unclear - verify first"
    else:
        clinical_use = "Check terms"

    print(f"{model_name:<35} {license_str:<25} {clinical_use}")

print("\n--- Notes ---")
print("Apache-2.0 and MIT licenses are permissive and allow clinical/commercial use.")
print("Llama models have a community license that allows most uses but has restrictions.")
print("Always read the full license and consult legal before clinical deployment.")

---
### Stretch Challenge

Search the HuggingFace Hub for models specifically trained on **Arabic clinical text** or **Arabic biomedical data**. Are there any? What are their limitations?

In [ ]:
# Stretch: Search for Arabic medical models
print("=== Arabic Medical Models ===")
arabic_models = api.list_models(search="arabic medical", sort="downloads", limit=10)
for model in arabic_models:
    print(f"  Model: {model.id}")
    print(f"  Downloads: {model.downloads}")
    print(f"  Tags: {model.tags[:5] if model.tags else 'N/A'}")
    print()

print("\n=== Arabic Clinical Models ===")
arabic_clinical = api.list_models(search="arabic clinical", sort="downloads", limit=10)
for model in arabic_clinical:
    print(f"  Model: {model.id}")
    print(f"  Downloads: {model.downloads}")
    print()

### KHCC Connection
The ability to evaluate and select open-source models is essential for KHCC's AI Office (AIDI). Models that run locally ensure:
- **Patient data privacy** — no data leaves KHCC's infrastructure
- **Cost efficiency** — no per-token API charges
- **Customization** — models can be fine-tuned on KHCC's own clinical data
- **Regulatory compliance** — full control over model versioning and audit trails